# Data Preprocessing Notebook (Decision Tree)
## CS 171 Final Project: West Coast Swing Dance Pattern Classification

**Author:** Nguyen Pham

This notebook handles data preprocessing for the Decision Tree model:
- Download YouTube dance videos
- Extract video frames
- Extract MediaPipe pose keypoints
- Prepare data for trick classification (Sugar Push vs Sugar Tag)

**Imports**

In [1]:
import os
from pathlib import Path
import pandas as pd
import yt_dlp
import re
from urllib.parse import urlparse, parse_qs
import sys

sys.path.insert(0, str(Path.cwd().parent))
from scripts.extract_frames import extract_frames
from scripts.extract_keypoints import extract_keypoints

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

**Download Videos**

In [2]:
def download_video(url, name, output_path, start_time=0, duration=4):
    # Combine output path with filename
    output_path = output_path / f'{name}.mp4'
    
    ydl_opts = {
        'format': 'best',
        'outtmpl': str(output_path),
        'quiet': True,
        'no_warnings': True,
        'download_ranges': lambda info_dict, ydl: [
            {
                'start_time': start_time,
                'end_time': start_time + duration,
            }
        ],
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
            print(f"Downloaded video: {name}")
        return True
    except Exception as e:
        print(f"Error downloading video: {e}")
        return False

**Extract Youtube ID**

In [3]:
def extract_youtube_id(url):
    if not url:
        return None
    
    # Parse URL
    parsed = urlparse(url)

    patterns = [
        r'(?:youtube\.com\/shorts\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/shorts/VIDEO_ID
        r'(?:youtube\.com\/embed\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/embed/VIDEO_ID
        r'(?:youtube\.com\/v\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/v/VIDEO_ID
        r'(?:youtu\.be\/)([a-zA-Z0-9_-]{11})', # youtu.be/VIDEO_ID
        r'(?:youtube\.com\/watch\?v=)([a-zA-Z0-9_-]{11})', # www.youtube.com/watch?v=VIDEO_ID
        r'(?:youtube\.com\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/VIDEO_ID
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            print("Found video ID:", match.group(1))
            return match.group(1)

    print("No video ID found in URL:", url)
    return None
    

**Create Labels**

In [4]:
def add_label(csv_path, url,start_time=0, duration=10, action_class=None, dance_style=None, labels=None, division=None, pattern=None):
    # Check if the CSV file exists
    if not os.path.exists(csv_path):
        df = pd.DataFrame(columns=['id', 'youtube_id', 'start_time', 'duration', 'action_class', 'dance_style','labels', 'division', 'pattern'])
        df.to_csv(csv_path, index=False)
    else:
        # Load existing CSV
        df = pd.read_csv(csv_path)

    # Get next ID
    if len(df) == 0:
        next_id = 1
    else:
        next_id = df['id'].max() + 1

    # Extract ID from URL
    video_id = extract_youtube_id(url)
    
    # Add new rows
    new_row = {
        'id': next_id,
        'youtube_id': video_id,
        'start_time': start_time,
        'duration': duration,
        'action_class': action_class,
        'dance_style': dance_style,
        'labels': labels,
        'division': division,
        'pattern': pattern,
    }

    # Append new rows
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    # Save updated CSV
    df.to_csv(csv_path, index=False)
    print(f"Added video with ID {next_id} to {csv_path}")

In [5]:
os.makedirs('../data', exist_ok=True)
CSV_PATH = '../data/hp.csv'

**Batch Download Function**

In [6]:
def download_all_from_csv(csv_path, folder):
    df = pd.read_csv(csv_path)

    for index, row in df.iterrows():
        # Create folder with skill level (division)
        os.makedirs(f'../data/raw/{folder}/{row["division"]}', exist_ok=True)

        # Download video
        video_name = f"{row['division']}_{row['id']}"
        VIDEO_DIR = Path(f'../data/raw/{folder}/{row["division"]}')
        
        print(f"Downloading {row['id']}: {row['division']}...")
        download_video(
            url=row['youtube_id'],
            name=video_name,
            output_path=VIDEO_DIR,
            start_time=row['start_time'],
            duration=row['duration'],
        )

In [7]:
# Download all videos
CSV_PATH = '../data/hp.csv'
download_all_from_csv(CSV_PATH, "videos_1")

Downloaded video: intermediate_1
Downloaded video: intermediate_1
Downloaded video: novice_2   
Downloaded video: novice_2
Downloaded video: intermediate_3
Downloaded video: intermediate_3
Downloaded video: intermediate_4
Downloaded video: intermediate_4
Downloaded video: intermediate_5
Downloaded video: intermediate_5
Downloaded video: advanced_6 
Downloaded video: advanced_6
Downloaded video: advanced_7 
Downloaded video: advanced_7
Downloaded video: intermediate_8
Downloaded video: intermediate_8
Downloaded video: intermediate_9
Downloaded video: intermediate_9
Downloaded video: advanced_10
Downloaded video: advanced_10
Downloaded video: advanced_11
Downloaded video: advanced_11
Downloaded video: advanced_12
Downloaded video: advanced_12
Downloaded video: intermediate_13
Downloaded video: intermediate_13
Downloaded video: advanced_14
Downloaded video: advanced_14
Downloaded video: advanced_15
Downloaded video: advanced_15
Downloaded video: intermediate_16
Downloaded video: intermedi

In [8]:
extract_frames(input_dir="../data/raw/videos_1",output_dir="../data/frames_1")
extract_keypoints(input_dir="../data/frames_1", output_dir="../data/keypoints_1", static_mode=True)

Found 22 mp4 files under ../data/raw/videos_1
→ ../data/raw/videos_1/advanced/advanced_6.mp4
   out: ../data/frames_1/advanced/advanced_6
   saved 13 frames
→ ../data/raw/videos_1/advanced/advanced_7.mp4
   out: ../data/frames_1/advanced/advanced_7
   saved 13 frames
→ ../data/raw/videos_1/advanced/advanced_7.mp4
   out: ../data/frames_1/advanced/advanced_7
   saved 12 frames
→ ../data/raw/videos_1/advanced/advanced_12.mp4
   out: ../data/frames_1/advanced/advanced_12
   saved 12 frames
→ ../data/raw/videos_1/advanced/advanced_12.mp4
   out: ../data/frames_1/advanced/advanced_12
   saved 15 frames
→ ../data/raw/videos_1/advanced/advanced_11.mp4
   out: ../data/frames_1/advanced/advanced_11
   saved 0 frames
→ ../data/raw/videos_1/advanced/advanced_10.mp4
   out: ../data/frames_1/advanced/advanced_10
   saved 15 frames
→ ../data/raw/videos_1/advanced/advanced_11.mp4
   out: ../data/frames_1/advanced/advanced_11
   saved 0 frames
→ ../data/raw/videos_1/advanced/advanced_10.mp4
   out: ..

I0000 00:00:1764663240.871629  685190 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


   saved 48 frames
Done!
Initializing MediaPipe Pose...
Found 22 video directories.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
  0%|          | 0/22 [00:00<?, ?it/s]W0000 00:00:1764663240.924107  687743 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764663240.983971  687750 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764663241.040856  687749 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
W0000 00:00:1764663240.924107  687743 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1764663240.983971  687750 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling s

Extraction complete.


{'advanced/advanced_6': {'num_frames': 13,
  'num_detected': 13,
  'npy_path': '../data/keypoints_1/advanced/advanced_6/keypoints.npy',
  'json_path': '../data/keypoints_1/advanced/advanced_6/keypoints.json'},
 'advanced/advanced_7': {'num_frames': 12,
  'num_detected': 12,
  'npy_path': '../data/keypoints_1/advanced/advanced_7/keypoints.npy',
  'json_path': '../data/keypoints_1/advanced/advanced_7/keypoints.json'},
 'advanced/advanced_12': {'num_frames': 15,
  'num_detected': 15,
  'npy_path': '../data/keypoints_1/advanced/advanced_12/keypoints.npy',
  'json_path': '../data/keypoints_1/advanced/advanced_12/keypoints.json'},
 'advanced/advanced_15': {'num_frames': 69,
  'num_detected': 69,
  'npy_path': '../data/keypoints_1/advanced/advanced_15/keypoints.npy',
  'json_path': '../data/keypoints_1/advanced/advanced_15/keypoints.json'},
 'advanced/advanced_14': {'num_frames': 18,
  'num_detected': 18,
  'npy_path': '../data/keypoints_1/advanced/advanced_14/keypoints.npy',
  'json_path': '